In [1]:
base_dir  = r"C:\Users\nalla\PycharmProjects\FlowerClassfication\flower_images"
print(f'Updated base_dir: {base_dir}')

Updated base_dir: C:\Users\nalla\PycharmProjects\FlowerClassfication\flower_images


In [2]:
import os
if os.path.exists(base_dir):
    print(f'Contents of "{base_dir}":')
    for item in os.listdir(base_dir):
        print(item)
else:
    print(f'The directory "{base_dir}" does not exist.')

Contents of "C:\Users\nalla\PycharmProjects\FlowerClassfication\flower_images":
Lilly
Lotus
Orchid
Sunflower
Tulip


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import tensorflow
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout,Activation
import os
base_dir = r"C:\Users\nalla\PycharmProjects\FlowerClassfication\flower_images"
img_size = 244
batch_size = 64

In [6]:
   # Check if the base directory exists
if not os.path.exists(base_dir):
    print(f'Error: The directory "{base_dir}" does not exist.Please ensure Google Drive is mounted and the correct path is provided.')
else:
  train_datagen = ImageDataGenerator(rescale=1./255,zoom_range=0.2,horizontal_flip= True, validation_split=0.2)
  test_datagen = ImageDataGenerator(rescale=1./255,validation_split=0.2)

  #Creating the datasets4
  train_datagen = train_datagen.flow_from_directory(base_dir,target_size=(img_size,img_size),subset='training',batch_size = batch_size)

  test_datagen = test_datagen.flow_from_directory(base_dir,target_size=(img_size,img_size),subset='validation',batch_size = batch_size)

# Model Development
  model = Sequential()
  model.add(Conv2D(filters=64,kernel_size=(5,5),padding='same',
  activation = 'relu',input_shape=(244,244,3)))
  model.add(MaxPooling2D(pool_size=(2,2)))
  model.add(Conv2D(filters=32,kernel_size=(5,5),padding='same',
  activation = 'relu'))
  model.add(MaxPooling2D(pool_size=(2,2)))
  model.add(Conv2D(filters=16,kernel_size=(5,5),padding='same',
  activation = 'relu'))
  model.add(MaxPooling2D(pool_size=(2,2)))
  model.add(Conv2D(filters=8,kernel_size=(5,5),padding='same',
  activation = 'relu'))
  model.add(MaxPooling2D(pool_size=(2,2)))
  model.add(Flatten())
  model.add(Dense(512,activation='relu'))
  model.add(Dense(5,activation = 'softmax')) #changed activation to softmax

  model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

  model.fit(train_datagen,epochs = 15,validation_data = test_datagen)

Found 4000 images belonging to 5 classes.
Found 1000 images belonging to 5 classes.


C:\Users\nalla\PycharmProjects\FlowerClassfication\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 114s 2s/step - accuracy: 0.3990 - loss: 1.3912 - val_accuracy: 0.4880 - val_loss: 1.2149
Epoch 2/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.4922 - loss: 1.2182 - val_accuracy: 0.5450 - val_loss: 1.0905
Epoch 3/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.5717 - loss: 1.0552 - val_accuracy: 0.5600 - val_loss: 1.1119
Epoch 4/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 95s 2s/step - accuracy: 0.6037 - loss: 0.9920 - val_accuracy: 0.6240 - val_loss: 0.9688
Epoch 5/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.6423 - loss: 0.9001 - val_accuracy: 0.6420 - val_loss: 0.8918
Epoch 6/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.6655 - loss: 0.8532 - val_accuracy: 0.6690 - val_loss: 0.8876
Epoch 7/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 78s 1s/step - accuracy: 0.6973 - loss: 0.7666 - val_accuracy: 0.6720 - val_loss: 0.8402
Epoch 8/15
63/63 ━━━━━━━━━━━━━━━━━━━━ 76s 1s/step - accuracy: 0.7303 - loss: 0.6981 - val_accuracy: 0.6930 - val_loss

In [7]:
import gradio as gr
import numpy as np
from PIL import Image
from tensorflow.keras.models import load_model

C:\Users\nalla\PycharmProjects\FlowerClassfication\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class_names = list(train_datagen.class_indices.keys())

def predict_image(img):
    # Resize the image to the model's expected input size
    img = img.resize((img_size, img_size))
    # Convert the image to a NumPy array
    img_array = np.array(img)
    # Normalize pixel values to the range [0, 1]
    img_array = img_array / 255.0
    # Add a batch dimension (e.g., from (224, 224, 3) to (1, 224, 224, 3))
    img_array = np.expand_dims(img_array, axis=0)

    # Make prediction
    predictions = model.predict(img_array)
    # Get the index of the predicted class
    predicted_class_index = np.argmax(predictions, axis=1)[0]
    # Get the class name
    predicted_class_name = class_names[predicted_class_index]

    return predicted_class_name

print("Predict function 'predict_image' defined successfully.")
print(f"Class names extracted from train_datagen: {class_names}")

Predict function 'predict_image' defined successfully.
Class names extracted from train_datagen: ['Lilly', 'Lotus', 'Orchid', 'Sunflower', 'Tulip']


In [ ]:
iface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Textbox(label="Predicted Class"),
    title="Flower Image Classifier",
    description="Upload an image of a flower to get its predicted class (daisy, lily, rose, sunflower, or tulips)."
)

iface.launch(debug=True, share=True)
print("Gradio interface launched successfully.")

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://f43007a01d095aa7c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
